# 🇪🇬 دبلجة تلقائية للفيديو - TBATE Hindi → Arabic

**فيديو:** The Beginning After The End Chapter 254 Explained in Hindi
**الرابط الأصلي:** https://youtu.be/KBBEelzYZYQ

## 📋 خطوات التشغيل
1. **Runtime → Change runtime type → T4 GPU**
2. اضغط **Run all**
3. في الخلية 2: **ارفع الفيديو من جهازك** (يدوياً)
4. انتظر 5-8 دقايق أول مرة
5. هتلقي رابط الواجهة في آخر الخلية 7
6. في الواجهة: اضغط **🚀 بدء الدبلجة التلقائية**

---

## ⚙️ إعدادات تلقائية:
- ✅ الفيديو بيتحمّل من جهازك (ارفع يدوياً)
- ✅ مقطع صوتي بيتقطع تلقائي للاستنساخ من الفيديو
- ✅ الترجمة: Hindi → Arabic (MSA)
- ✅ الصوت: استنساخ صوت المتكلم الأصلي
- ✅ مزامنة شفاه تلقائية

---

## 💡 لو عايز لهجة مصرية بدل الفصحى:
في الخلية 6 غير `TARGET_LANG = "ar"` إلى `TARGET_LANG = "ar-EG"`

---

## ⏱️ الوقت المتوقع:
- أول تشغيل: 5-8 دقايق (تحميل النماذج)
- دبلجة 7 دقايق فيديو: 10-15 دقيقة

---

## 📌 ملاحظات:
- Colab بيقطع الجلسة بعد 12 ساعة أو idle
- الفيديو المدبلج هيتحفظ في Google Drive تلقائي
- لو حصلت مشكلة، restart Runtime و Run all تاني

---

## 🎯 طريقة الرفع:
في الخلية 2 هتلقي زر **Choose Files** — اختار الفيديو من جهازك واضغط Upload

In [ ]:
# ── 1. فحص GPU ────────────────────────────────────────────────────────────
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    print("✅ GPU detected")
else:
    print("⚠️ مفيش GPU! اذهب لـ Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── 2. رفع الفيديو يدوياً ─────────────────────────────────────────────────
from google.colab import files
import os, glob

DOWNLOAD_DIR = "/content/videos"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Check if video already exists in /content/
existing_videos = (
    glob.glob("/content/*.mp4") +
    glob.glob("/content/*.webm") +
    glob.glob("/content/*.mkv") +
    glob.glob("/content/*.mov") +
    glob.glob("/content/*.avi")
)

if existing_videos:
    print("✅ Found existing video in /content/:")
    for v in existing_videos:
        print(f"   📹 {os.path.basename(v)} ({os.path.getsize(v)/1024/1024:.1f} MB)")
    # Move to DOWNLOAD_DIR
    for v in existing_videos:
        os.replace(v, os.path.join(DOWNLOAD_DIR, os.path.basename(v)))
else:
    print("📤 ارفع الفيديو من جهازك:")
    print("   - اضغط Choose Files واختار الفيديو")
    print("   - أو اسحب الملف هنا")
    print("   - انتظر الرفع ثم اضغط Run على الخلية التالية")

    uploaded = files.upload()

    # Move uploaded files to DOWNLOAD_DIR
    for filename in uploaded.keys():
        source = os.path.join("/content", filename)
        dest = os.path.join(DOWNLOAD_DIR, filename)
        os.replace(source, dest)
        print(f"✅ Uploaded: {filename}")

# List all videos
video_files = (
    glob.glob(f"{DOWNLOAD_DIR}/*.mp4") +
    glob.glob(f"{DOWNLOAD_DIR}/*.webm") +
    glob.glob(f"{DOWNLOAD_DIR}/*.mkv") +
    glob.glob(f"{DOWNLOAD_DIR}/*.mov") +
    glob.glob(f"{DOWNLOAD_DIR}/*.avi")
)

if not video_files:
    raise SystemExit("❌ مفيش فيديو! ارفع الفيديو أولاً")

video_path = video_files[0]
print(f"\n✅ Video ready: {video_path}")
for f in video_files:
    size = os.path.getsize(f)
    print(f"   📹 {os.path.basename(f)} ({size/1024/1024:.1f} MB)")

In [ ]:
# ── 3. استخراج مقطع صوتي للاستنساخ ────────────────────────────────────────
import glob

# Find the uploaded video
video_files = (
    glob.glob(f"{DOWNLOAD_DIR}/*.mp4") +
    glob.glob(f"{DOWNLOAD_DIR}/*.webm") +
    glob.glob(f"{DOWNLOAD_DIR}/*.mkv") +
    glob.glob(f"{DOWNLOAD_DIR}/*.mov") +
    glob.glob(f"{DOWNLOAD_DIR}/*.avi")
)
if not video_files:
    raise SystemExit("❌ No video file found! ارفع الفيديو في الخلية 2 أولاً")

video_path = video_files[0]
print(f"📹 Found video: {video_path}")

# Extract full audio from video
audio_path = f"{DOWNLOAD_DIR}/full_audio.wav"
subprocess.run([
    "ffmpeg", "-y", "-i", video_path,
    "-vn",  # no video
    "-acodec", "pcm_s16le",
    "-ar", "24000",
    "-ac", "1",  # mono
    audio_path
], check=True)

# Extract 10-second reference clip (from 30s mark)
ref_audio = "/content/reference_voice.wav"
subprocess.run([
    "ffmpeg", "-y",
    "-ss", "30",  # start at 30 seconds
    "-t", "10",   # 10 seconds
    "-i", audio_path,
    "-ar", "24000",
    "-ac", "1",
    ref_audio
], check=True)

print(f"\n✅ Reference audio extracted: {ref_audio}")
print("   📢 This is a 10-second clip from the original speaker")
print("   🎯 It will be used to clone the voice for Arabic dubbing")
    
# Check audio levels
result = subprocess.run([
    "ffmpeg", "-i", ref_audio,
    "-af", "volumedetect",
    "-f", "null", "-"
], capture_output=True, text=True)
for line in result.stderr.split('\n'):
    if 'mean_volume' in line or 'max_volume' in line:
        print(f"   {line.strip()}")

In [ ]:
# ── 4. تحميل وتثبيت VoiceStudio ───────────────────────────────────────────
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/debpalash/VoiceStudio.git"
REPO_DIR = "/content/VoiceStudio"
BUN = "/root/.bun/bin/bun"

os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}")
    rc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode
    if rc != 0:
        raise SystemExit(f"\n❌ FAILED: {what or shown} (exit code {rc})\nScroll up for error")

# Clone repo
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo already present — updating")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

# System packages
run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get")

# Install bun
if not os.path.exists(BUN):
    run("curl -fsSL https://bun.sh/install | bash", what="bun installer")
os.environ["PATH"] = os.path.dirname(BUN) + os.pathsep + os.environ["PATH"]
run([BUN, "--version"], what="bun version check")

# Build frontend
dist_index = os.path.join(REPO_DIR, "frontend", "dist", "index.html")
if not os.path.exists(dist_index):
    run([BUN, "install", "--frozen-lockfile"], cwd=REPO_DIR, what="bun install")
    run([BUN, "run", "--cwd", "electron", "build:web"], cwd=REPO_DIR, what="build frontend")
    if not os.path.exists(dist_index):
        raise SystemExit("Frontend build failed")
else:
    print("Frontend already built — skipping")

# Install backend
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
run([
    "uv", "pip", "install", "--system", "--no-cache",
    "--constraint", "deploy/torch-constraints.txt",
    "--override", "deploy/uv-overrides.txt",
    "."
], cwd=REPO_DIR, what="backend install")

# cuDNN setup
os.makedirs(os.path.join(REPO_DIR, ".venv"), exist_ok=True)
run([sys.executable, os.path.join(REPO_DIR, "scripts", "setup.py")], what="cuDNN setup")

# Sanity check
run([sys.executable, "-c",
    "import torch, torchaudio, uvicorn, fastapi, transformers; "
    "print(f'torch {torch.__version__}, torchaudio {torchaudio.__version__}, "
    "transformers {transformers.__version__}, CUDA: {torch.cuda.is_available()}'); "
    "from omnivoice.models.omnivoice import OmniVoice; "
    "from transformers import HiggsAudioV2TokenizerModel; "
    "print('✅ Install OK - backend ready')"
], cwd=REPO_DIR, what="import sanity check")

print("\n✅ VoiceStudio installed successfully!")

In [ ]:
# ── 5. إعداد Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Set data directory to Drive
os.environ['OMNIVOICE_DATA_DIR'] = '/content/drive/MyDrive/omnivoice_data'
DATA_DIR = os.environ['OMNIVOICE_DATA_DIR']
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'outputs'), exist_ok=True)

# Copy reference audio to data dir
import shutil
ref_audio_dest = os.path.join(DATA_DIR, 'reference_voice.wav')
shutil.copy(ref_audio, ref_audio_dest)

print(f"✅ Data dir: {DATA_DIR}")
print(f"✅ Reference voice copied: {ref_audio_dest}")

In [ ]:
# ── 6. تشغيل Backend ───────────────────────────────────────────────────────
import requests, json, threading, time

BACKEND_PORT = 3900
BASE = f"http://127.0.0.1:{BACKEND_PORT}"
LOG_FILE = "/content/omnivoice_backend.log"

# Start backend
backend_proc = subprocess.Popen(
    [sys.executable, "-m", "omnivoice.backend", "--port", str(BACKEND_PORT), "--data-dir", DATA_DIR],
    cwd=REPO_DIR,
    stdout=open(LOG_FILE, "w"),
    stderr=subprocess.STDOUT,
    text=True
)
print(f"Backend PID: {backend_proc.pid}")

# Wait for backend
print("Waiting for backend to start...")
for i in range(60):
    try:
        r = requests.get(f"{BASE}/health", timeout=2)
        if r.status_code == 200:
            print(f"\n✅ Backend is up! ({BASE})")
            break
    except:
        pass
    time.sleep(2)
else:
        raise SystemExit("❌ Backend failed to start. Check log at /content/omnivoice_backend.log")

print("\n🎯 Backend health:", requests.get(f"{BASE}/health").json())

In [ ]:
# ── 7. تشغيل الواجهة + دبلجة تلقائية ──────────────────────────────────────
import gradio as gr

# Auto-dub function
def auto_dub_video(video_path, target_lang="ar", voice_mode="clone"):
    """Automatically dub the video using VoiceStudio API"""
    import requests, json, time
    
    base_url = "http://127.0.0.1:3900"
    
    # Step 1: Upload video
    print("📤 جاري رفع الفيديو...")
    with open(video_path, "rb") as f:
        r = requests.post(
            f"{base_url}/dub/upload",
            files={"video": (os.path.basename(video_path), f, "video/mp4")},
            data={"input_type": "video"},
            timeout=300
        )
    
    if r.status_code != 202:
        return None, f"❌ Upload failed: {r.text}"
    
    job_id = r.json()["job_id"]
    prep_task = r.json()["task_id"]
    print(f"✅ Uploaded! Job ID: {job_id}")
    
    # Step 2: Transcribe
    print("🎧 جاري تحويل الصوت لنص...")
    r = requests.post(f"{base_url}/dub/transcribe/{job_id}", timeout=1800)
    if r.status_code != 200:
        return None, f"❌ Transcription failed: {r.text}"
    segments = r.json()["segments"]
    print(f"✅ Transcribed {len(segments)} segments")
    
    # Step 3: Translate
    print("🌍 جاري الترجمة للعربية...")
    r = requests.post(f"{base_url}/dub/translate", json={
        "job_id": job_id,
        "target_lang": target_lang,
        "provider": "google",  # Free Google Translate
        "segments": [{
            "id": str(s["id"]),
            "text": s["text"],
            "start": s["start"],
            "end": s["end"]
        } for s in segments]
    }, timeout=600)
    
    if r.status_code != 200:
        return None, f"❌ Translation failed: {r.text}"
    
    translated = {t["id"]: t["text"] for t in r.json()["translated"]}
    print(f"✅ Translated {len(translated)} segments")
    
    # Step 4: Generate dubbed audio
    print("🎙️ جاري توليد الصوت العربي...")
    r = requests.post(f"{base_url}/dub/generate/{job_id}", json={
        "language": "Arabic",
        "language_code": target_lang,
        "segments": [{
            "start": s["start"],
            "end": s["end"],
            "text": translated.get(str(s["id"]), s["text"])
        } for s in segments],
        "voice_clone": True,
        "reference_audio": ref_audio_dest
    }, timeout=120)
    
    if r.status_code != 200:
        return None, f"❌ Audio generation failed: {r.text}"
    
    task_id = r.json()["task_id"]
    print(f"🎨 Generating dubbed audio (task {task_id})...")
    
    # Poll for completion
    for _ in range(120):  # 10 minutes max
        time.sleep(5)
        r = requests.get(f"{base_url}/jobs/{task_id}", timeout=30)
        if r.status_code == 200:
            status = r.json().get("status")
            if status == "done":
                break
            if status in ("failed", "cancelled"):
                return None, f"❌ Dubbing {status}: {r.text}"
    else:
        return None, "❌ Dubbing timed out"
    
    # Step 5: Download dubbed video
    print("📥 جاري تحميل الفيديو المدبلج...")
    r = requests.get(f"{base_url}/dub/download/{job_id}",
        params={"include_tracks": target_lang, "default_track": target_lang},
        timeout=1800)
    
    if r.status_code != 200:
        return None, f"❌ Download failed: {r.text}"
    
    output_path = os.path.join(DATA_DIR, "outputs", "dubbed_video.mp4")
    with open(output_path, "wb") as f:
        f.write(r.content)
    
    print(f"\n🎉 Done! Video saved to: {output_path}")
    return output_path, "✅ تمت الدبلجة بنجاح! الفيديو جاهز"

# Build Gradio UI
with gr.Blocks(title="🇪🇬 TBATE Arabic Dub", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🇪🇬 دبلجة TBATE Chapter 254 - للعربية\n\n**فيديو:** The Beginning After The End Chapter 254 Explained in Hindi")
    
    with gr.Row():
        lang_choice = gr.Radio(
            choices=["ar (فصحى)", "ar-EG (مصرية)"],
            value="ar (فصحى)",
            label="🌍 اللغة الهدف"
        )
    
    dub_btn = gr.Button("🚀 بدء الدبلجة التلقائية", variant="primary", size="lg")
    output_video = gr.Video(label="🎬 الفيديو المدبلج")
    status = gr.Textbox(label="📊 الحالة", interactive=False, lines=3)
    
    def on_dub(lang_choice):
        lang_code = "ar" if "فصحى" in lang_choice else "ar-EG"
        status.value = "⏳ جاري الرفع والمعالجة... (ده بياخد 10-15 دقيقة)"
        output_path, msg = auto_dub_video(video_path, target_lang=lang_code)
        if output_path:
            return output_path, msg
        return None, msg
    
    dub_btn.click(
        on_dub,
        inputs=[lang_choice],
        outputs=[output_video, status]
    )

# Launch
print("\n" + "="*60)
print("🚀 launching interface...")
demo.launch(share=True, debug=True)

---

## 🎯 بعد ما تشغل Run all:

1. **الخلية 2:** لو لقيت فيها رسالة "✅ Found existing video" يعني الفيديو بيتعرف تلقائي
2. **انتظر 5-8 دقايق** أول مرة (تحميل النماذج)
3. هتلقي **رابط واجهة ويب** في آخر الخلية 7 => اضغط عليه
4. في الواجهة:
   - اختر اللغة: **ar (فصحى)** أو **ar-EG (مصرية)**
   - اضغط **🚀 بدء الدبلجة التلقائية**
5. **استنى 10-15 دقيقة** والدبلجة هتتم تلقائي
6. هتلقي الفيديو المدبلج في آخر الواجهة

---

## 📂 حفظ النتيجة:
- الفيديو المدبلج محفوظ في Google Drive تلقائي
- من Colab: Files → Drive → MyDrive → omnivoice_data → outputs → dubbed_video.mp4

---

## ⚠️ لو حصلت مشكلة:
1. **Backend مش شغال؟** شغل الخلية 6 لوحدها تاني
2. **الواجهة مش فاتحة؟** تأكد من T4 GPU وتشغيل الخلية 7
3. **الدبلجة فشلت؟** شغل الخلية 7 تاني (البيانات محفوظة في Drive)
4. **انقطع الاتصال؟** reconnect Colab و Run all من الخلية 5 (البيانات في Drive)